In [ ]:
!pip install yfinance lightgbm xgboost scikit-learn pandas numpy requests joblib

import numpy as np
import pandas as pd
import yfinance as yf
import requests
import time
import joblib
from datetime import datetime, timedelta

In [ ]:
LATITUDE = 17.7368
LONGITUDE = 83.3185

stocks = ["NTPC.NS", "TATAPOWER.NS", "ADANIPOWER.NS", "JSWENERGY.NS", "NHPC.NS"]

company_type_map = {
    "NTPC.NS": "thermal",
    "ADANIPOWER.NS": "thermal",
    "TATAPOWER.NS": "mixed",
    "JSWENERGY.NS": "mixed",
    "NHPC.NS": "hydro"
}

In [ ]:
def get_stock_data(ticker):
    df = yf.download(ticker, period="10y", interval="1d", progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    return df[['Close']].reset_index()

In [ ]:
def get_nifty():
    df = yf.download("^NSEI", period="10y", interval="1d", progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df[['Close']].reset_index()
    df.rename(columns={'Close': 'Nifty'}, inplace=True)
    return df

In [ ]:
def get_weather_forecast():
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "daily": "temperature_2m_mean,precipitation_sum",
        "forecast_days": 7,
        "timezone": "auto"
    }

    res = requests.get(url, params=params)
    data = res.json()

    temps = data["daily"]["temperature_2m_mean"]
    rain = data["daily"]["precipitation_sum"]

    return temps, rain

In [ ]:
weather_df = get_weather()
nifty_df = get_nifty()

/tmp/ipykernel_8279/3850100406.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("^NSEI", period="10y", interval="1d", progress=False)


In [ ]:
all_data = []

for stock in stocks:
    stock_df = get_stock_data(stock)

    merged = stock_df.merge(weather_df, on="Date", how="inner")
    merged = merged.merge(nifty_df, on="Date", how="inner")

    merged["company"] = stock
    merged["company_type"] = merged["company"].map(company_type_map)

    all_data.append(merged)

df = pd.concat(all_data).sort_values("Date").reset_index(drop=True)

/tmp/ipykernel_8279/2056005263.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", interval="1d", progress=False)
/tmp/ipykernel_8279/2056005263.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", interval="1d", progress=False)
/tmp/ipykernel_8279/2056005263.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", interval="1d", progress=False)
/tmp/ipykernel_8279/2056005263.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", interval="1d", progress=False)
/tmp/ipykernel_8279/2056005263.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="10y", interval="1d", progress=False)


In [ ]:
df['return'] = df['Close'].pct_change() * 100

df['lag1'] = df['return'].shift(1)
df['lag2'] = df['return'].shift(2)
df['lag3'] = df['return'].shift(3)

df['market_return'] = df['Nifty'].pct_change() * 100

df['temp_change'] = df['Temp'].diff()
df['temp_lag1'] = df['Temp'].shift(1)
df['temp_lag2'] = df['Temp'].shift(2)

df['heat_intensity'] = (df['Temp'] - 32).clip(lower=0)
df['strong_heat'] = (df['Temp'] > 33).astype(int)

df['rolling_temp_mean'] = df['Temp'].rolling(7).mean()
df['rolling_temp_std'] = df['Temp'].rolling(7).std()

df['rain_lag1'] = df['Rain'].shift(1)
df['rain_rolling'] = df['Rain'].rolling(7).mean()
df['heavy_rain'] = (df['Rain'] > 20).astype(int)

df['rain_x_company'] = df['Rain'] * df['company_type']
# DEMAND PROXY
df['demand_signal'] = df['Temp'] * 0.7 + df['temp_change'] * 0.3
df['demand_change'] = df['demand_signal'].diff()
df['demand_spike'] = (df['demand_change'] > df['demand_change'].quantile(0.75)).astype(int)

df['volatility'] = df['return'].rolling(5).std()
df['momentum'] = df['Close'].pct_change(3) * 100

df['temp_x_market'] = df['temp_change'] * df['market_return']
df['temp_x_demand'] = df['temp_change'] * df['demand_change']


# TARGET
df['future_return'] = df['return'].shift(-5)
df['target'] = 0
df.loc[df['future_return'] > 0.5, 'target'] = 1

df = df.dropna().reset_index(drop=True)

In [ ]:
features = [
    'temp_change','temp_lag1','temp_lag2','heat_intensity','strong_heat',
    'rolling_temp_mean','rolling_temp_std',
    'demand_change','demand_spike',
    'market_return',
    'lag1','lag2','lag3',
    'volatility','momentum',
    'temp_x_market','temp_x_demand',
    'company_type'
]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {}
accuracies = {}

for stock in stocks:
    temp_df = df[df['company'] == stock]

    X = temp_df[features]
    y = temp_df['target']

    split = int(len(X) * 0.8)

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)

    models[stock] = model
    accuracies[stock] = acc

    print(stock, "Accuracy:", acc)

print("Average:", np.mean(list(accuracies.values())))

NTPC.NS Accuracy: 0.5212981744421906
TATAPOWER.NS Accuracy: 0.4959349593495935
ADANIPOWER.NS Accuracy: 0.5273833671399595
JSWENERGY.NS Accuracy: 0.537525354969574
NHPC.NS Accuracy: 0.5496957403651116
Average: 0.526367519253286


In [ ]:
def climate_adjustment(temp, rain, company):

    ctype = company_type_map[company]
    score = 0

    # 🔥 Heat impact
    if temp > 33:
        if ctype == "thermal":
            score += 1
        elif ctype == "hydro":
            score -= 1

    # 🌧️ Rain impact
    if rain > 15:
        if ctype == "hydro":
            score += 1

    return score

In [ ]:
def generate_explanation(temp, rain, company):

    ctype = company_type_map[company]

    if temp > 33:
        if ctype == "thermal":
            return "High temperature increases electricity demand, benefiting thermal power companies."
        elif ctype == "hydro":
            return "High temperature may reduce water levels, negatively affecting hydro generation."

    if rain > 15:
        if ctype == "hydro":
            return "Increased rainfall boosts hydroelectric generation, supporting NHPC performance."

    return "Stable climate conditions suggest steady electricity demand with limited volatility."

In [ ]:
def get_trade_plan(signal):

    if signal == "BUY":
        return {
            "entry": "Next trading session",
            "holding": "3–5 days"
        }
    else:
        return {
            "entry": "Wait for stronger signal",
            "holding": "No active trade"
        }

In [ ]:
for stock in models:
    joblib.dump(models[stock], f"{stock}.pkl")

joblib.dump(features, "features.pkl")

print("✅ Saved all models")

✅ Saved all models
